In [ ]:
import Pkg
Pkg.add("JuMP")
Pkg.add("HiGHS")

# Student Teacher Meetings

In [15]:
# agiso@dtu.dk
using JuMP, HiGHS

StudentTeacherMeetings=[
 0 1 0 0 1 1;
 0 0 1 0 1 1;
 1 0 0 1 0 0;
 0 1 1 0 0 0;
 1 0 0 1 0 0;
 1 1 0 1 0 0;
 1 1 1 0 0 0;
 0 0 0 0 1 1;
 0 0 0 1 1 1;
 0 0 0 1 1 1;
 0 0 1 0 1 1;
 0 1 0 0 1 1;
 0 1 1 0 0 0;
 0 1 1 0 0 0;
 1 0 0 1 1 0
]
S=15
T=6
TS=11

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

@variable(model, x[1:S, 1:T, 1:TS], Bin)
@variable(model, LS[1:S] >= 0) # Latest time slot for each student

# We want to sum up all the latest time slots
@objective(model, Min, sum(LS[s] for s in 1:S))

# Every meeting takes place once if requsted
@constraint(model, [s in 1:S, t in 1:T], sum(x[s, t, ts] for ts in 1:TS) == StudentTeacherMeetings[s, t])

# Each teacher can have one student per time slot
@constraint(model, [t in 1:T, ts in 1:TS], sum(x[s, t, ts] for s in 1:S) <= 1)

# Each student can have one teacher per time slot
@constraint(model, [s in 1:S, ts in 1:TS], sum(x[s, t, ts] for t in 1:T) <= 1)


# Set LS[s] to be the highest time slot
@constraint(model, [s in 1:S, t in 1:T, k in 1:TS],
    LS[s] >= k * x[s,t,k]
)


optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("Times: ", (value.(LS)))
println("Times: ", (value.(x)))


Optimal solution:
z = 65.99999999997583
Times: [5.0, 4.0, 2.0, 3.000000000016523, 2.0000000000001026, 4.999999999999936, 7.0, 2.0, 8.0, 5.99999999996948, 6.0, 7.0, 2.9999999999897855, 2.0, 4.000000000000006]
Times: [0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0 0.0 0.9999999999898759; -0.0 0.0 0.0 1.0 0.0 0.0; 0.0 0.0 0.9999999999897281 0.0 0.0 0.0; 0.9999999999999344 0.0 0.0 0.0 0.0 0.0; 6.561418075534675e-14 -0.0 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0 1.0 0.0; 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0 0.0 1.0124123761556802e-11; 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 1.0214669104769722e-11 0.0 0.0 0.0; 0.0 1.0 5.417888360170735e-14 0.0 0.0 0.0; -0.0 0.0 0.0 -0.0 0.0 0.0;;; 0.0 0.0 0.0 0.0 0.0 -0.0; 0.0 0.0 0.0 0.0 -0.0 1.0123679672346952e-11; 0.9999999999999566 0.0 0.0 4.340972026284362e-14 0.0 0.0; 0.0 0.0 4.829470157119431e-14 0.0 0.0 0.0; 0.0 0.0 0.0 0.9999999999999633 0.0 0.0; 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0 0.0 0.0; 0.0 0.0 0.0 0.0 -0.0 0

# Workplan for Teaching Assistants


Part 1

In [26]:
using JuMP, HiGHS

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)


include("Data/WorkplanData.jl")

@variable(model, x[1:TA, 1:P, 1:D], Bin)

@objective(model, Min,
    sum(x[ta, p, d] * Inconvenience[ta, p, d] for ta in 1:TA, p in 1:P, d in 1:D)
)

# Demand constraint
@constraint(model, [p in 1:P, d in 1:D], sum(x[ta, p, d] for ta in 1:TA) == Demand[p, d])

# Each TA has to work 52 hours 
@constraint(model, [ta in 1:TA], sum(x[ta, p, d] for p in 1:P, d in 1:D) == 52)



optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))



Optimal solution:
z = 94.97538677918425


Part 2 : When you inspect the generated plan, you observe that the TAs work several times on
each day. This is rather inconvenient. You decide to modify your model and include the
restrictions that each TA can only work consecutive hours and that if a TA works on a
particular day, then the TA has to work at least two hours.

In [ ]:
using JuMP, HiGHS

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)


include("Data/WorkplanData.jl")

@variable(model, x[1:TA, 1:P, 1:D], Bin) # Woking plan
@variable(model, y[1:TA, 1:P, 1:D], Bin) # When a TA starts working
@variable(model, working_hours[1:TA, 1:D] >= 0) # When a TA starts working

@objective(model, Min,
    sum(x[ta, p, d] * Inconvenience[ta, p, d] for ta in 1:TA, p in 1:P, d in 1:D)
)

# Demand constraint
@constraint(model, [p in 1:P, d in 1:D], sum(x[ta, p, d] for ta in 1:TA) == Demand[p, d])

# Each TA has to work 52 hours 
@constraint(model, [ta in 1:TA], sum(x[ta, p, d] for p in 1:P, d in 1:D) == 52)

###### ------ Adding consecutivess hours ----- #####
# TA can only start work once a day
@constraint(model, [ta in 1:TA, d in 1:D],
    sum(y[ta, p, d] for p in 1:P) <= 1
)

# Ta can only work if he worked the previous hour or if just started working
@constraint(model, [ta in 1:TA, p in 1:P, d in 1:D],
    x[ta, p, d] <= (p > 1 ? x[ta, p - 1, d] : 0) + y[ta, p, d]
)

### At least two consecutive hours
# Start by defining working hours
@constraint(model, [ta in 1:TA, d in 1:D],
    working_hours[ta, d] == sum(x[ta, p, d] for p in 1:P)
)

# Set working hours greter than 2
@constraint(model, [ta in 1:TA, d in 1:D],
    working_hours[ta, d] >= 2 * sum(y[ta, p, d] for p in 1:P)
)

optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("working_hours = ", (value.(working_hours)))



Optimal solution:
z = 121.97081575246132
working_hours = [8.0 3.0 3.0 6.0 8.0 7.0 3.0 6.0 8.0; 8.0 8.0 6.0 2.0 8.0 6.0 8.0 6.0 0.0; 8.0 8.0 8.0 6.0 3.0 6.0 8.0 2.0 3.0; 6.0 8.0 8.0 6.0 6.0 6.0 6.0 6.0 0.0]


# Scrap removal

In [42]:
using JuMP, HiGHS


weights = [
    35 10 45 53 37 22 26 38 63 17 44 54 62 42 39 51 24 52 46 29
]

I = 20
B = 10

price = 50
limit = 100

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)


@variable(model, bag_used[1:B], Bin)
@variable(model, not_used[1:B], Bin)
@variable(model, x[1:B, 1:I], Bin)


@objective(model, Min,  sum(bag_used[b] * price for b in 1:B))

@constraint(model, [b in 1:B], 
    sum(x[b, i] * weights[i] for i in 1:I) <= bag_used[b] * limit
)

@constraint(model, [i in 1:I],
    sum(x[b,i] for b in 1:B) == 1
)

@constraint(model, 
    sum(x[b, i] for b in 1:B, i in 1:I) == I
)

@constraint(model, [b in 1:B],
    not_used[b]  == 1 - bag_used[b]
)



optimize!(model)

println("Optimal solution:")
println("z = ", (objective_value(model)))

println("\nSales:")
println(value.(bag_used))
println(value.(not_used))
for b in 1:B
    println(value.(x[b,:]))
end
println(sum(value.(x[b,i]) for b in 1:B, i in 1:I) )

println("\nHow packed are they?")
for b in 1:B
    println(sum(value.(x[b,i]) * weights[i] for i in 1:I))
end

Optimal solution:
z = 399.99999999999994

Sales:
[0.0, 0.0, 0.9999999999999988, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 1.2212453270876722e-15, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, 0.0, 0.0]
[0.0, -0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.9984014443252818e-15, 0.0, 0.0, -2.4424906541753444e-15, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.885780586188048e-15, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.9999999999999949, 0.0, 0.0, 0.0, -0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[5.548527542801849e-15, 0.0, -0.0, 5.067397445112707e-15, 0.0, 0.0, -0.0, 0.0, -0.0, 0.0, -5.9893429565242275e-15, 0.0, 0.0, -3.885767984586222e-15, 0.0, 0.0, 0.0, 0.9999999999999961, 1.0000000000000002, 0.0]
[0.9999999999999953, 0.0, 0.0, 0.0, 0.0, 3.1258902984284348e-15, 0.0, 0.0, 0.9999999999999987, 0.0, 0.0, 1.3800722103716315e-15, -0.0, 1.3800722103716315e-15, 0.0, 0.0, -0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0